# DeepGaze IIE — Google Colab Training Pipeline
**Paper:** *Calibrated prediction in and out-of-domain for state-of-the-art saliency modeling*  
**Authors:** Linardos, Kümmerer, Press, Bethge (2021)

---
## Kiến trúc tổng quan
```
Image → [Backbone CNN ×4 | FROZEN] → [Readout 1×1 Conv | TRAINABLE] → [Finalizer | TRAINABLE] → Log-density Map
```
**Ensemble:** DenseNet201 + ResNext50, mỗi backbone 10 components → logsumexp average

---
### Cấu trúc Drive cần có
```
MyDrive/Data PBL4/
    ALLSTIMULI.zip   ← ảnh gốc MIT1003 (1003 ảnh .jpeg)
    MIT.zip          ← fixation data MIT1003 (per subject .mat)
    Salicon.zip      ← SALICON dataset (ảnh + fixation .mat + maps)
    checkpoints/     ← tự tạo khi train
```
### Thứ tự chạy
1. Cell 1 → Cell 2 → Cell 3 → ... → Cell 11 (setup)
2. Cell 12 (Phase 1: SALICON pretrain)
3. Cell 13 (Phase 2: MIT1003 finetune)
4. Cell 14 (Visualize)


---
## Cell 1 — Cài đặt thư viện

In [1]:
# ============================================================
# CELL 1: CÀI ĐẶT THƯ VIỆN
# Colab có Internet mặc định — chỉ chạy 1 lần mỗi session
# ============================================================

import subprocess, sys

def pip_install(pkg):
    r = subprocess.run([sys.executable,'-m','pip','install',pkg,'-q'],
                       capture_output=True, text=True)
    print(f'{'✅' if r.returncode==0 else '❌'} {pkg}')
    if r.returncode != 0: print(r.stderr[-200:])

pip_install('pysaliency')            # quản lý dataset saliency
pip_install('boltons')               # utility
pip_install('efficientnet_pytorch')  # EfficientNet-B5 backbone

import torch
if torch.cuda.is_available():
    print(f'\n🚀 GPU: {torch.cuda.get_device_name(0)}')
    print(f'   VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')
else:
    print('\n⚠️  GPU chưa bật! → Runtime → Change runtime type → GPU')

print('\n✅ Cell 1 hoàn tất')


✅ pysaliency
✅ boltons
✅ efficientnet_pytorch

🚀 GPU: Tesla T4
   VRAM: 15.6 GB

✅ Cell 1 hoàn tất


---
## Cell 2 — Mount Drive & Copy Dataset vào local SSD
> Copy vào `/content/` trước khi train: I/O nhanh hơn Drive **10-50x**  
> Dùng **stream copy** (từng chunk 64MB) để tránh lỗi `Transport endpoint` với file lớn

In [2]:
# ============================================================
# CELL 2: MOUNT DRIVE & COPY DATASET
# Stream copy tránh lỗi 'Transport endpoint is not connected'
# khi copy file lớn (Salicon.zip ~4GB) từ Drive
# ============================================================

from google.colab import drive
import os, zipfile, time

# 2.1 Mount Drive (force_remount=True để làm mới kết nối)
print('Mounting Google Drive...')
drive.mount('/content/drive', force_remount=True)
print('✅ Drive mounted')

# 2.2 Đường dẫn nguồn
DRIVE_DIR      = '/content/drive/MyDrive/Data PBL4'
MIT_ZIP        = os.path.join(DRIVE_DIR, 'MIT.zip')
SALICON_ZIP    = os.path.join(DRIVE_DIR, 'Salicon.zip')
STIMULI_ZIP    = os.path.join(DRIVE_DIR, 'ALLSTIMULI.zip')

for name, f in [('MIT.zip', MIT_ZIP), ('Salicon.zip', SALICON_ZIP), ('ALLSTIMULI.zip', STIMULI_ZIP)]:
    mb = os.path.getsize(f)/1e6 if os.path.exists(f) else -1
    print(f'  {'✅' if mb>0 else '❌'} {name}: {mb:.0f} MB')

# 2.3 Thư mục đích
LOCAL_DATA = '/content/data'
os.makedirs(LOCAL_DATA, exist_ok=True)

# 2.4 Stream copy có retry
def stream_copy(src, dst, chunk_mb=64, max_retry=3):
    chunk    = chunk_mb * 1024 * 1024
    src_size = os.path.getsize(src)
    dst_size = os.path.getsize(dst) if os.path.exists(dst) else 0
    if dst_size == src_size:
        print('    (đã có local copy, bỏ qua)')
        return
    for attempt in range(1, max_retry+1):
        try:
            copied, t0 = 0, time.time()
            with open(src,'rb') as fsrc, open(dst,'wb') as fdst:
                while True:
                    buf = fsrc.read(chunk)
                    if not buf: break
                    fdst.write(buf)
                    copied += len(buf)
                    pct   = copied/src_size*100
                    speed = copied/(time.time()-t0)/1e6
                    print(f'    {pct:5.1f}%  {speed:.0f} MB/s', end='\r', flush=True)
            print(f'    100.0%  done in {time.time()-t0:.1f}s          ')
            return
        except OSError as e:
            print(f'\n    ⚠️  Lần {attempt} lỗi: {e}')
            if attempt < max_retry:
                drive.mount('/content/drive', force_remount=True)
                time.sleep(3)
            else:
                raise RuntimeError(f'Copy thất bại sau {max_retry} lần') from e

def copy_and_extract(zip_src, extract_dst, label):
    if os.path.exists(extract_dst) and os.listdir(extract_dst):
        print(f'⏭️  {label}: đã giải nén, bỏ qua')
        return
    os.makedirs(extract_dst, exist_ok=True)
    tmp = f'/content/{label}.zip'
    print(f'📦 Copying {label} ({os.path.getsize(zip_src)/1e6:.0f} MB)...')
    stream_copy(zip_src, tmp)
    print(f'📂 Giải nén {label}...', end=' ', flush=True)
    t0 = time.time()
    with zipfile.ZipFile(tmp,'r') as zf:
        zf.extractall(extract_dst)
    os.remove(tmp)
    n = sum(len(files) for _,_,files in os.walk(extract_dst))
    print(f'{time.time()-t0:.1f}s  ({n} files)')

copy_and_extract(MIT_ZIP,     '/content/data/MIT',     'MIT')
copy_and_extract(SALICON_ZIP, '/content/data/Salicon', 'Salicon')
copy_and_extract(STIMULI_ZIP, '/content/data/ALLSTIMULI', 'ALLSTIMULI')

print('\n✅ Cell 2 hoàn tất')


Mounting Google Drive...
Mounted at /content/drive
✅ Drive mounted
  ✅ MIT.zip: 47 MB
  ✅ Salicon.zip: 4264 MB
  ✅ ALLSTIMULI.zip: 235 MB
📦 Copying MIT (47 MB)...
    100.0%  done in 1.2s          
📂 Giải nén MIT... 3.3s  (30172 files)
📦 Copying Salicon (4264 MB)...
    100.0%  done in 58.6s          
📂 Giải nén Salicon... 49.3s  (55000 files)
📦 Copying ALLSTIMULI (235 MB)...
    100.0%  done in 3.0s          
📂 Giải nén ALLSTIMULI... 1.7s  (3013 files)

✅ Cell 2 hoàn tất


---
## Cell 3 — Cấu hình đường dẫn & Hyperparameters

In [3]:
# ============================================================
# CELL 3: CẤU HÌNH — THAY ĐỔI EXPERIMENT_NAME ĐỂ CHẠY TỔ HỢP KHÁC
#
# 7 tổ hợp backbone cần test:
# ── 2 trong 4 (C(4,2) = 6 tổ hợp) ──
#   'SN_EN'  : ShapeNetC + EfficientNetB5      (index 0,1)
#   'SN_DN'  : ShapeNetC + DenseNet201         (index 0,2)
#   'SN_RX'  : ShapeNetC + ResNext50           (index 0,3)
#   'EN_DN'  : EfficientNetB5 + DenseNet201    (index 1,2)
#   'EN_RX'  : EfficientNetB5 + ResNext50      (index 1,3)
#   'DN_RX'  : DenseNet201 + ResNext50         (index 2,3)  ← đã chạy
# ── 3 trong 4 (C(4,3) = 4 tổ hợp) ──
#   'SN_EN_DN' : ShapeNetC + EfficientNetB5 + DenseNet201   (index 0,1,2)
#   'SN_EN_RX' : ShapeNetC + EfficientNetB5 + ResNext50     (index 0,1,3)
#   'SN_DN_RX' : ShapeNetC + DenseNet201 + ResNext50        (index 0,2,3)
#   'EN_DN_RX' : EfficientNetB5 + DenseNet201 + ResNext50   (index 1,2,3)
# ============================================================

import os, torch

# ── THAY DÒNG NÀY TRƯỚC KHI CHẠY ────────────────────────
EXPERIMENT_NAME    = 'SN_EN_DN'          # tên thư mục lưu checkpoint
BACKBONE_INDICES   = [0,1,2]          # index backbone tương ứng
# ─────────────────────────────────────────────────────────
# Bảng tra cứu nhanh:
# EXPERIMENT_NAME = 'SN_EN'  → BACKBONE_INDICES = [0, 1]
# EXPERIMENT_NAME = 'SN_DN'  → BACKBONE_INDICES = [0, 2]
# EXPERIMENT_NAME = 'SN_RX'  → BACKBONE_INDICES = [0, 3]
# EXPERIMENT_NAME = 'EN_DN'  → BACKBONE_INDICES = [1, 2]
# EXPERIMENT_NAME = 'EN_RX'  → BACKBONE_INDICES = [1, 3]
# EXPERIMENT_NAME = 'DN_RX'  → BACKBONE_INDICES = [2, 3]
# EXPERIMENT_NAME = 'SN_EN_DN' → BACKBONE_INDICES = [0,1,2]
# EXPERIMENT_NAME = 'SN_EN_RX' → BACKBONE_INDICES = [0,1,3]
# EXPERIMENT_NAME = 'SN_DN_RX' → BACKBONE_INDICES = [0,2,3]
# EXPERIMENT_NAME = 'EN_DN_RX' → BACKBONE_INDICES = [1,2,3]

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device          : {DEVICE}')
print(f'Experiment      : {EXPERIMENT_NAME}')
print(f'Backbone indices: {BACKBONE_INDICES}')

# ─── Đường dẫn dataset ───────────────────────────────────
SAL_IMG_TRAIN = '/content/data/Salicon/images/images/train'
SAL_IMG_VAL   = '/content/data/Salicon/images/images/val'
SAL_FIX_TRAIN = '/content/data/Salicon/fixations/train'
SAL_FIX_VAL   = '/content/data/Salicon/fixations/val'
MIT_DATA_DIR  = '/content/data/MIT/DATA'
MIT_SUBJECTS  = ['CNG','ajs','emb','ems','ff','hp','jcw','jw',
                 'kae','krl','po','tmj','tu','ya','zb']
MIT_STIM_DIR  = '/content/data/ALLSTIMULI/ALLSTIMULI'

# ─── Checkpoint riêng cho từng experiment ────────────────
# Mỗi tổ hợp backbone có folder riêng trên Drive
DRIVE_BASE = '/content/drive/MyDrive/Data PBL4/experiments'
DRIVE_CKPT = os.path.join(DRIVE_BASE, EXPERIMENT_NAME)
LOCAL_CKPT = f'/content/checkpoints_{EXPERIMENT_NAME}'
os.makedirs(DRIVE_CKPT, exist_ok=True)
os.makedirs(LOCAL_CKPT,  exist_ok=True)

print(f'Drive checkpoint: {DRIVE_CKPT}')
print(f'Local checkpoint: {LOCAL_CKPT}')

# Kiểm tra đường dẫn
print('\nKiểm tra đường dẫn dataset:')
all_ok = True
for name, path in [
    ('SAL_IMG_TRAIN', SAL_IMG_TRAIN), ('SAL_IMG_VAL',  SAL_IMG_VAL),
    ('SAL_FIX_TRAIN', SAL_FIX_TRAIN), ('SAL_FIX_VAL', SAL_FIX_VAL),
    ('MIT_DATA_DIR',  MIT_DATA_DIR),  ('MIT_STIM_DIR', MIT_STIM_DIR),
]:
    ok = os.path.exists(path)
    n  = len(os.listdir(path)) if ok else 0
    print(f'  {"✅" if ok else "❌"} {name}: {n} items')
    if not ok: all_ok = False

# ─── Hyperparameters ─────────────────────────────────────
CFG = {
    'lr_pretrain'        : 1e-3,
    'lr_finetune'        : 1e-3,
    'lr_decay_factor'    : 0.1,
    'min_lr'             : 1e-6,
    'batch_size'         : 2,
    'num_workers'        : 0,
    'downsample'         : 2,
    'readout_factor'     : 16,
    'saliency_map_factor': 2,
    'initial_sigma'      : 8.0,
    'n_components'       : 10,
    'ds_salicon'         : 1.5,
    'ds_mit'             : 2.0,
    'local_ckpt'         : LOCAL_CKPT,
    'drive_ckpt'         : DRIVE_CKPT,
    'experiment'         : EXPERIMENT_NAME,
    'backbone_indices'   : BACKBONE_INDICES,
}
print(f'\n✅ Cell 3 hoàn tất — Experiment: {EXPERIMENT_NAME}')


Device          : cuda
Experiment      : SN_EN_DN
Backbone indices: [0, 1, 2]
Drive checkpoint: /content/drive/MyDrive/Data PBL4/experiments/SN_EN_DN
Local checkpoint: /content/checkpoints_SN_EN_DN

Kiểm tra đường dẫn dataset:
  ✅ SAL_IMG_TRAIN: 10000 items
  ✅ SAL_IMG_VAL: 5000 items
  ✅ SAL_FIX_TRAIN: 10000 items
  ✅ SAL_FIX_VAL: 5000 items
  ✅ MIT_DATA_DIR: 63 items
  ✅ MIT_STIM_DIR: 1005 items

✅ Cell 3 hoàn tất — Experiment: SN_EN_DN


---
## Cell 4 — Custom Layers

In [4]:
# ============================================================
# CELL 4: CUSTOM LAYERS (giữ nguyên từ code gốc tác giả)
# GaussianFilterNd : learnable blur trong Finalizer
# LayerNorm        : normalize spatial feature map (B,C,H,W)
# Bias             : bias layer tách khỏi Conv
# ============================================================

import math
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from collections import OrderedDict


def gaussian_filter_1d(tensor, dim, sigma, truncate=4, kernel_size=None,
                        padding_mode='replicate', padding_value=0.0):
    sigma = torch.as_tensor(sigma, device=tensor.device, dtype=tensor.dtype)
    if kernel_size is None:
        kernel_size = torch.as_tensor(
            2*torch.ceil(truncate*sigma)+1, device=tensor.device, dtype=torch.int64)
    kernel_size     = kernel_size.detach()
    kernel_size_int = kernel_size.detach().cpu().numpy()
    mean   = (torch.as_tensor(kernel_size, dtype=tensor.dtype)-1)/2
    grid   = (torch.arange(kernel_size, device=tensor.device)-mean).view(1,1,-1).detach()
    src    = tensor.shape
    tensor = torch.movedim(tensor, dim, len(src)-1)
    dls    = tensor.shape
    tensor = tensor.reshape(-1, 1, src[dim])
    pad    = (math.ceil((kernel_size_int-1)/2),)*2
    t_     = F.pad(tensor, pad, padding_mode, padding_value)
    kernel = torch.exp(-0.5*(grid/sigma)**2)
    kernel = kernel/kernel.sum()
    t_     = F.conv1d(t_, kernel)
    t_     = t_.view(dls)
    return torch.movedim(t_, len(src)-1, dim)


class GaussianFilterNd(nn.Module):
    def __init__(self, dims, sigma, truncate=4, trainable=False,
                 padding_mode='replicate', padding_value=0.0, kernel_size=None):
        super().__init__()
        self.dims          = dims
        self.sigma         = nn.Parameter(torch.tensor(sigma, dtype=torch.float32),
                                           requires_grad=trainable)
        self.truncate      = truncate
        self.kernel_size   = kernel_size
        self.padding_mode  = padding_mode
        self.padding_value = padding_value

    def forward(self, x):
        for dim in self.dims:
            x = gaussian_filter_1d(x, dim=dim, sigma=self.sigma,
                                    truncate=self.truncate, kernel_size=self.kernel_size,
                                    padding_mode=self.padding_mode,
                                    padding_value=self.padding_value)
        return x


class LayerNorm(nn.Module):
    """LayerNorm cho spatial map (B,C,H,W). Normalize trên (C,H,W)."""
    def __init__(self, features, eps=1e-12):
        super().__init__()
        self.features = features
        self.eps      = eps
        self.weight   = nn.Parameter(torch.ones(features))
        self.bias     = nn.Parameter(torch.zeros(features))

    def forward(self, x):
        h, w   = x.shape[2], x.shape[3]
        weight = self.weight.view(-1,1,1).expand(-1,h,w).contiguous()
        bias   = self.bias.view(-1,1,1).expand(-1,h,w).contiguous()
        return F.layer_norm(x, (self.features,h,w), weight, bias, self.eps)


class Bias(nn.Module):
    def __init__(self, channels):
        super().__init__()
        self.bias = nn.Parameter(torch.zeros(channels))
    def forward(self, x):
        return x + self.bias[None,:,None,None]


class LayerNormMultiInput(nn.Module):
    def __init__(self, features, eps=1e-12):
        super().__init__()
        self.features = features
        for k, f in enumerate(features):
            if f: setattr(self, f'ln{k}', LayerNorm(f, eps=eps))
    def forward(self, tensors):
        return [getattr(self,f'ln{k}')(t) if f else None
                for k,(f,t) in enumerate(zip(self.features, tensors))]


class Conv2dMultiInput(nn.Module):
    def __init__(self, in_channels, out_channels, kernel_size, bias=True):
        super().__init__()
        self.in_channels = in_channels
        for k, c in enumerate(in_channels):
            if c: setattr(self, f'conv{k}', nn.Conv2d(c, out_channels, kernel_size, bias=bias))
    def forward(self, tensors):
        out = None
        for k,(c,t) in enumerate(zip(self.in_channels, tensors)):
            if not c: continue
            _o = getattr(self,f'conv{k}')(t)
            out = _o if out is None else out+_o
        return out


print('✅ Custom layers OK')


✅ Custom layers OK


---
## Cell 5 — Normalizer & Backbone Classes

In [5]:
# ============================================================
# CELL 5: NORMALIZER & BACKBONE
# Mỗi backbone: Normalizer (ImageNet stats) + pretrained CNN
# Backbone weights bị FREEZE hoàn toàn khi train
# ============================================================

import torchvision
from torch.utils import model_zoo


class Normalizer(nn.Module):
    """Chuẩn hóa [0,255] → ImageNet mean/std. persistent=False."""
    def __init__(self):
        super().__init__()
        m = torch.tensor([0.485,0.456,0.406], dtype=torch.float32).view(3,1,1)
        s = torch.tensor([0.229,0.224,0.225], dtype=torch.float32).view(3,1,1)
        self.register_buffer('mean', m, persistent=False)
        self.register_buffer('std',  s, persistent=False)
    def forward(self, x):
        return (x/255.0 - self.mean) / self.std


def _load_shapenet_c():
    """ShapeNet-C: ResNet50 train SIN+IN → finetune IN.
    Checkpoint dùng DataParallel → wrap Sequential(module=...).
    """
    url = ('https://bitbucket.org/robert_geirhos/texture-vs-shape-pretrained-models'
           '/raw/60b770e128fffcbd8562a3ab3546c1a735432d03'
           '/resnet50_finetune_60_epochs_lr_decay_after_30_start_resnet50'
           '_train_45_epochs_combined_IN_SF-ca06340c.pth.tar')
    model = torchvision.models.resnet50(pretrained=False)
    model = nn.Sequential(OrderedDict([('module', model)]))
    ckpt  = model_zoo.load_url(url, map_location='cpu')
    model.load_state_dict(ckpt['state_dict'])
    return model

class RGBShapeNetC(nn.Sequential):
    def __init__(self):
        super().__init__(Normalizer(), _load_shapenet_c())

class RGBEfficientNetB5(nn.Sequential):
    def __init__(self):
        from efficientnet_pytorch import EfficientNet
        super().__init__(Normalizer(), EfficientNet.from_pretrained('efficientnet-b5'))

class RGBDenseNet201(nn.Sequential):
    def __init__(self):
        dn = torch.hub.load('pytorch/vision:v0.10.0','densenet201',pretrained=True)
        super().__init__(Normalizer(), dn)

class RGBResNext50(nn.Sequential):
    def __init__(self):
        rx = torch.hub.load('pytorch/vision:v0.10.0','resnext50_32x4d',pretrained=True)
        super().__init__(Normalizer(), rx)


print('✅ Backbone classes OK')


✅ Backbone classes OK


---
## Cell 6 — Core Modules

In [6]:
# ============================================================
# CELL 6: CORE MODULES
# FeatureExtractor : forward hook lấy intermediate features
# Finalizer        : blur + centerbias + log-softmax → log-density
# DeepGazeIIIMixture: ensemble nhiều readout / 1 backbone
# MixtureModel     : ensemble nhiều backbone (inter-model complementarity)
# ============================================================


class FeatureExtractor(nn.Module):
    """Hook vào backbone, lấy output tại các target layers."""
    def __init__(self, features, targets):
        super().__init__()
        self.features = features
        self.targets  = targets
        self.outputs  = {}
        for t in targets:
            layer = dict([*self.features.named_modules()])[t]
            layer.register_forward_hook(self._hook(t))

    def _hook(self, lid):
        def fn(_,__,out): self.outputs[lid] = out.clone()
        return fn

    def forward(self, x):
        self.outputs.clear()
        self.features(x)
        return [self.outputs[t] for t in self.targets]


class Finalizer(nn.Module):
    """readout → resize → blur → +centerbias → upsample → log-softmax.
    Trainable: sigma (blur), center_bias_weight.
    """
    def __init__(self, sigma=8.0, learn_sigma=True,
                 center_bias_weight=1.0, learn_center_bias_weight=True,
                 saliency_map_factor=2):
        super().__init__()
        self.smf   = saliency_map_factor
        self.gauss = GaussianFilterNd([2,3], sigma, truncate=3, trainable=learn_sigma)
        self.cbw   = nn.Parameter(torch.tensor([center_bias_weight]),
                                   requires_grad=learn_center_bias_weight)

    def forward(self, readout, centerbias):
        cb  = F.interpolate(centerbias.unsqueeze(1),
                            scale_factor=1/self.smf,
                            recompute_scale_factor=False)[:,0]
        out = F.interpolate(readout, size=[cb.shape[1], cb.shape[2]])
        out = self.gauss(out)
        out = out[:,0]
        out = out + self.cbw * cb
        out = F.interpolate(out.unsqueeze(1),
                            size=[centerbias.shape[1], centerbias.shape[2]])[:,0]
        out = out - out.logsumexp(dim=(1,2), keepdim=True)
        return out


class DeepGazeIIIMixture(nn.Module):
    """Nhiều readout networks chia sẻ 1 backbone (frozen).
    Backbone luôn ở eval() kể cả khi model.train().
    """
    def __init__(self, features, saliency_networks, scanpath_networks,
                 fixation_selection_networks, finalizers,
                 downsample=2, readout_factor=16, saliency_map_factor=2,
                 included_fixations=None):
        super().__init__()
        self.downsample  = downsample
        self.rf          = readout_factor
        self.smf         = saliency_map_factor
        self.features    = features
        for p in self.features.parameters(): p.requires_grad = False
        self.features.eval()
        self.sal_nets = nn.ModuleList(saliency_networks)
        self.scn_nets = nn.ModuleList(scanpath_networks)
        self.fix_nets = nn.ModuleList(fixation_selection_networks)
        self.fins     = nn.ModuleList(finalizers)

    def forward(self, x, centerbias, **kw):
        orig = x.shape
        xd   = F.interpolate(x, scale_factor=1/self.downsample,
                              recompute_scale_factor=False)
        with torch.no_grad():
            feats = self.features(xd)
        rs    = [math.ceil(orig[2]/self.downsample/self.rf),
                 math.ceil(orig[3]/self.downsample/self.rf)]
        ri    = torch.cat([F.interpolate(f,rs) for f in feats], dim=1)
        preds = []
        for sal,scn,fix,fin in zip(self.sal_nets,self.scn_nets,self.fix_nets,self.fins):
            s = sal(ri)
            s = fix((s, None))
            s = fin(s, centerbias)
            preds.append(s.unsqueeze(1))
        preds = torch.cat(preds, dim=1) - np.log(len(self.sal_nets))
        return preds.logsumexp(dim=1, keepdim=True)

    def train(self, mode=True):
        self.features.eval()
        for m in [*self.sal_nets, *self.fix_nets, *self.fins]:
            m.train(mode)


class MixtureModel(nn.Module):
    """Ensemble nhiều backbone models (inter-model complementarity)."""
    def __init__(self, models):
        super().__init__()
        self.models = nn.ModuleList(models)
    def forward(self, x, centerbias, **kw):
        preds = torch.cat([m(x,centerbias) for m in self.models], dim=1)
        preds -= np.log(len(self.models))
        return preds.logsumexp(dim=1, keepdim=True)


print('✅ Core modules OK')


✅ Core modules OK


---
## Cell 7 — Build DeepGaze IIE

In [7]:
# ============================================================
# CELL 7: BUILD DeepGaze IIE
# Readout: 3 lớp 1×1 Conv + LayerNorm + Softplus
# 1×1 kernel: kết hợp channels, không tạo spatial features mới
# ============================================================

BACKBONES_CFG = [
    {
        'name':'ShapeNetC', 'class':RGBShapeNetC,
        'used_features':['1.module.layer3.0.conv2','1.module.layer3.3.conv2',
                         '1.module.layer3.5.conv1','1.module.layer3.5.conv2',
                         '1.module.layer4.1.conv2','1.module.layer4.2.conv2'],
        'channels':2048,
    },
    {
        'name':'EfficientNetB5', 'class':RGBEfficientNetB5,
        'used_features':['1._blocks.24._depthwise_conv',
                         '1._blocks.26._depthwise_conv',
                         '1._blocks.35._project_conv'],
        'channels':2416,
    },
    {
        'name':'DenseNet201', 'class':RGBDenseNet201,
        'used_features':['1.features.denseblock4.denselayer32.norm1',
                         '1.features.denseblock4.denselayer32.conv1',
                         '1.features.denseblock4.denselayer31.conv2'],
        'channels':2048,
    },
    {
        'name':'ResNext50', 'class':RGBResNext50,
        'used_features':['1.layer3.5.conv1','1.layer3.5.conv2',
                         '1.layer3.4.conv2','1.layer4.2.conv2'],
        'channels':2560,
    },
]


def build_saliency_network(C):
    return nn.Sequential(OrderedDict([
        ('ln0',nn.Sequential(LayerNorm(C))),
        ('ln0_',LayerNorm(C)),
        ('c0', nn.Conv2d(C,  8,(1,1),bias=False)), ('b0',Bias(8)),  ('sp0',nn.Softplus()),
        ('ln1',LayerNorm(8)),
        ('c1', nn.Conv2d(8, 16,(1,1),bias=False)), ('b1',Bias(16)), ('sp1',nn.Softplus()),
        ('ln2',LayerNorm(16)),
        ('c2', nn.Conv2d(16, 1,(1,1),bias=False)), ('b2',Bias(1)),  ('sp2',nn.Softplus()),
    ]))


def build_saliency_network(C):
    """3-layer 1x1 Conv readout. LayerNorm + Softplus."""
    return nn.Sequential(OrderedDict([
        ('layernorm0', LayerNorm(C)),
        ('conv0',      nn.Conv2d(C,  8,(1,1),bias=False)),
        ('bias0',      Bias(8)),
        ('softplus0',  nn.Softplus()),
        ('layernorm1', LayerNorm(8)),
        ('conv1',      nn.Conv2d(8, 16,(1,1),bias=False)),
        ('bias1',      Bias(16)),
        ('softplus1',  nn.Softplus()),
        ('layernorm2', LayerNorm(16)),
        ('conv2',      nn.Conv2d(16, 1,(1,1),bias=False)),
        ('bias2',      Bias(1)),
        ('softplus3',  nn.Softplus()),
    ]))


def build_fixation_selection_network():
    """Kết hợp saliency + scanpath (None với IIE)."""
    return nn.Sequential(OrderedDict([
        ('layernorm0', LayerNormMultiInput([1,0])),
        ('conv0',      Conv2dMultiInput([1,0],128,(1,1),bias=False)),
        ('bias0',      Bias(128)),
        ('softplus0',  nn.Softplus()),
        ('layernorm1', LayerNorm(128)),
        ('conv1',      nn.Conv2d(128,16,(1,1),bias=False)),
        ('bias1',      Bias(16)),
        ('softplus1',  nn.Softplus()),
        ('conv2',      nn.Conv2d(16,1,(1,1),bias=False)),
    ]))


def build_backbone_model(cfg, n):
    backbone  = cfg['class']()
    extractor = FeatureExtractor(backbone, cfg['used_features'])
    C = cfg['channels']
    return DeepGazeIIIMixture(
        features=extractor,
        saliency_networks           = [build_saliency_network(C)          for _ in range(n)],
        scanpath_networks           = [None]*n,
        fixation_selection_networks = [build_fixation_selection_network() for _ in range(n)],
        finalizers                  = [Finalizer(sigma=CFG['initial_sigma'],
                                                  learn_sigma=True,
                                                  saliency_map_factor=CFG['saliency_map_factor']
                                                  ) for _ in range(n)],
        downsample=CFG['downsample'], readout_factor=CFG['readout_factor'],
        saliency_map_factor=CFG['saliency_map_factor'], included_fixations=[],
    )


def build_deepgaze_iie(backbone_indices=None, n_components=None):
    """
    backbone_indices=None      → tất cả 4 backbone (full paper)
    backbone_indices=[2,3]     → DenseNet201 + ResNext50 (khuyến nghị T4)
    backbone_indices=[2]       → chỉ DenseNet201 (debug)
    n_components=10            → 1 instance × 10 folds
    n_components=30            → 3 instances × 10 folds (paper)
    """
    n    = n_components or CFG['n_components']
    cfgs = BACKBONES_CFG if backbone_indices is None \
           else [BACKBONES_CFG[i] for i in backbone_indices]
    print(f'Building DeepGaze IIE:')
    print(f'  Backbones  : {[c["name"] for c in cfgs]}')
    print(f'  Components : {n} per backbone  |  Total: {len(cfgs)*n} sub-models')
    models = []
    for cfg in cfgs:
        print(f'  Loading {cfg["name"]}...', end=' ', flush=True)
        models.append(build_backbone_model(cfg, n))
        print('✅')
    model = MixtureModel(models)
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total     = sum(p.numel() for p in model.parameters())
    print(f'\nParameters: {trainable:,} trainable / {total:,} total')
    return model


print('✅ Build functions OK')


✅ Build functions OK


---
## Cell 8 — Metrics
> **Đơn vị: nats/fixation** (KHÔNG đổi sang bits trong tính toán)  
> `IG = LL_model - baseline_LL` — cả hai cùng đơn vị nats  
> Khi in ra mới đổi: `IG_bits = IG_nats / log(2)`

In [8]:
# ============================================================
# CELL 8: METRICS (v5 - FIXED NSS)
# BUG CU: fixation_mask co gia tri 1,2,3 (accumulate=True)
#         → NSS bi thoi phong len ~85
# FIX: mask phai la BINARY (0/1), NSS dung = 1.5 - 3.5
# IG khong bi anh huong (log_likelihood dung)
# ============================================================

import numpy as np
import torch


def log_likelihood(log_density, fixation_mask, weights=None):
    """Log-likelihood tai fixation points. Don vi: nats/fix.
    fixation_mask: binary (0/1) hoac count — deu dung vi chia fix_count.
    """
    if log_density.dim() == 4:
        log_density = log_density[:, 0]
    if weights is None:
        weights = torch.ones(log_density.shape[0], device=log_density.device)
    weights   = len(weights) * weights.view(-1,1,1) / weights.sum()
    fix_count = fixation_mask.sum(dim=(-1,-2), keepdim=True).clamp(min=1)
    return torch.mean(
        weights * torch.sum(log_density * fixation_mask,
                            dim=(-1,-2), keepdim=True) / fix_count
    )


def nss(log_density, fixation_mask, weights=None):
    """Normalized Scanpath Saliency (NSS).

    Cong thuc dung:
      1. density = exp(log_density)
      2. sal_map = (density - mean) / std   <- z-score toan anh
      3. NSS = mean(sal_map tai vi tri fixation)

    QUAN TRONG: fixation_mask phai BINARY (0/1).
    Neu mask co gia tri >1 (accumulate=True) thi NSS se bi thoi phong.
    Ket qua ky vong: NSS = 1.5 - 3.5 voi model tot.
    """
    if log_density.dim() == 4:
        log_density = log_density[:, 0]
    if weights is None:
        weights = torch.ones(log_density.shape[0], device=log_density.device)

    weights   = len(weights) * weights.view(-1,1,1) / weights.sum()

    # Binary mask: clip ve 0/1 de dam bao dung du mask co accumulate
    binary_mask = (fixation_mask > 0).float()
    fix_count   = binary_mask.sum(dim=(-1,-2), keepdim=True).clamp(min=1)

    # Tinh density tu log-density
    density = torch.exp(log_density)

    # Z-score normalize toan bo saliency map
    mean    = density.mean(dim=(-1,-2), keepdim=True)
    std     = density.std(dim=(-1,-2), keepdim=True)
    sal_map = (density - mean) / (std + 1e-8)

    # NSS = trung binh z-score tai vi tri fixation
    nss_per_image = torch.sum(sal_map * binary_mask,
                              dim=(-1,-2), keepdim=True) / fix_count
    return torch.mean(weights * nss_per_image)


def compute_baseline_ll(centerbias_template, fix_data, img_paths, downsample):
    """Baseline LL tu centerbias. Don vi: nats (KHONG chia log(2))."""
    from scipy.ndimage import zoom
    from scipy.special import logsumexp
    from PIL import Image
    lls = []
    for fp, fix in zip(img_paths, fix_data):
        try:
            w, h = Image.open(fp).size
        except Exception:
            continue
        h_ = int(h / downsample)
        w_ = int(w / downsample)
        cb = zoom(centerbias_template,
                  (h_/centerbias_template.shape[0], w_/centerbias_template.shape[1]),
                  order=0, mode='nearest')
        cb -= logsumexp(cb)
        xs = np.array(fix['x'], dtype=float)
        ys = np.array(fix['y'], dtype=float)
        # Loc NaN (MIT1003 eyeData co NaN)
        valid = ~(np.isnan(xs) | np.isnan(ys))
        xs, ys = xs[valid], ys[valid]
        xs = np.clip((xs/downsample).astype(int), 0, w_-1)
        ys = np.clip((ys/downsample).astype(int), 0, h_-1)
        if len(xs):
            lls.append(cb[ys, xs].mean())
    return float(np.mean(lls)) if lls else 0.0


print('Metrics OK (v5)')
print('NSS ky vong: 1.5 - 3.5 (model tot)')
print('IG  ky vong: 0.50-0.60 nats SALICON | 0.65-0.85 nats MIT1003')


Metrics OK (v5)
NSS ky vong: 1.5 - 3.5 (model tot)
IG  ky vong: 0.50-0.60 nats SALICON | 0.65-0.85 nats MIT1003


---
## Cell 9 — Dataset & DataLoader

In [9]:
# ============================================================
# CELL 9: DATASET & DATALOADER (v5)
# FIX 1: SaliencyDataset tao BINARY mask (0/1), loc NaN
#        - Truoc: accumulate=True → mask co gia tri 1,2,3 → NSS sai
#        - Sau : unique positions → mask binary → NSS dung
# FIX 2: _read_mit_mat doc dung format struct MIT1003
#        eyeData = struct.DATA.eyeData (N,2)
# ============================================================

import glob, os
import numpy as np
import scipy.io as sio
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from scipy.ndimage import zoom
from scipy.special import logsumexp
import torch
import torch.nn.functional as F


class SaliencyDataset(Dataset):
    def __init__(self, image_paths, fixation_data, centerbias_template, downsample=1.0):
        self.image_paths = image_paths
        self.fix_data    = fixation_data
        self.cb_tmpl     = centerbias_template
        self.ds          = downsample

    def __len__(self): return len(self.image_paths)

    def _cb(self, h, w):
        cb = zoom(self.cb_tmpl,
                  (h/self.cb_tmpl.shape[0], w/self.cb_tmpl.shape[1]),
                  order=0, mode='nearest')
        cb -= logsumexp(cb)
        return cb.astype(np.float32)

    def __getitem__(self, idx):
        img = np.array(Image.open(self.image_paths[idx]).convert('RGB'))
        if self.ds != 1.0:
            nh,nw = int(img.shape[0]/self.ds), int(img.shape[1]/self.ds)
            img   = np.array(Image.fromarray(img).resize((nw,nh), Image.BILINEAR))
        h, w    = img.shape[:2]
        image_t = torch.from_numpy(img.transpose(2,0,1).astype(np.float32))
        cb_t    = torch.from_numpy(self._cb(h, w))

        fix = self.fix_data[idx]
        xs  = np.array(fix['x'], dtype=float)
        ys  = np.array(fix['y'], dtype=float)

        # ── FIX: Loc NaN (MIT1003 eyeData co the co NaN) ────
        valid = ~(np.isnan(xs) | np.isnan(ys))
        xs, ys = xs[valid], ys[valid]

        if self.ds != 1.0:
            xs = xs / self.ds
            ys = ys / self.ds

        xs = np.clip(xs.astype(int), 0, w-1)
        ys = np.clip(ys.astype(int), 0, h-1)

        # ── FIX: Tao BINARY mask dung unique positions ───────
        # Truoc: accumulate=True → pixel nhin nhieu lan co gia tri 2,3...
        #        → NSS bi thoi phong (~85 thay vi ~2.5)
        # Sau:  unique positions → mask chi co 0 hoac 1
        mask = torch.zeros(h, w, dtype=torch.float32)
        n_fix = len(xs)
        if n_fix > 0:
            flat_idx = ys * w + xs                     # flatten 2D→1D
            unique_flat = np.unique(flat_idx)           # loai trung
            u_y = unique_flat // w
            u_x = unique_flat %  w
            mask[u_y, u_x] = 1.0                       # binary 0/1

        # weight = so fixation goc (truoc unique) de log_likelihood dung
        return {
            'image'        : image_t,
            'centerbias'   : cb_t,
            'fixation_mask': mask,
            'weight'       : torch.tensor(float(max(n_fix, 1))),
        }


def collate_fn(batch):
    max_h = max(b['image'].shape[1] for b in batch)
    max_w = max(b['image'].shape[2] for b in batch)
    imgs,cbs,masks,ws = [],[],[],[]
    for b in batch:
        h,w = b['image'].shape[1], b['image'].shape[2]
        ph,pw = max_h-h, max_w-w
        imgs.append( F.pad(b['image'],         [0,pw,0,ph]))
        cbs.append(  F.pad(b['centerbias'],    [0,pw,0,ph], value=float(b['centerbias'].min())))
        masks.append(F.pad(b['fixation_mask'], [0,pw,0,ph]))
        ws.append(b['weight'])
    return {
        'image'        : torch.stack(imgs),
        'centerbias'   : torch.stack(cbs),
        'fixation_mask': torch.stack(masks),
        'weight'       : torch.stack(ws),
    }


def _read_salicon_mat(mat_path):
    """SALICON: gaze[i].fixations = Nx2 [x,y]."""
    try:
        mat = sio.loadmat(mat_path, squeeze_me=True, struct_as_record=False)
    except Exception:
        return [], []
    xs, ys = [], []
    if 'gaze' in mat:
        try:
            for obs in np.atleast_1d(mat['gaze']):
                fix = np.atleast_2d(obs.fixations)
                if fix.shape[1] >= 2:
                    xs.extend(fix[:,0].tolist())
                    ys.extend(fix[:,1].tolist())
        except Exception:
            pass
    if not xs:
        for k,v in mat.items():
            if k.startswith('_'): continue
            if isinstance(v,np.ndarray) and v.ndim==2 and v.shape[1]>=2:
                xs,ys = v[:,0].tolist(), v[:,1].tolist()
                break
    return xs, ys


def _read_mit_mat(mat_path):
    """MIT1003 format (da xac nhan qua debug):
    key=imgID → struct { DATA → struct { eyeData: (N,2) float64 [x,y] } }
    """
    try:
        mat = sio.loadmat(mat_path, squeeze_me=True, struct_as_record=False)
    except Exception:
        return [], []
    xs, ys = [], []
    for k, v in mat.items():
        if k.startswith('_'): continue
        try:
            # v.DATA.eyeData = (N,2) [x, y]
            eye = v.DATA.eyeData
            eye = np.array(eye, dtype=float)
            eye = np.atleast_2d(eye)
            if eye.ndim == 2 and eye.shape[1] >= 2 and eye.shape[0] > 0:
                xs.extend(eye[:,0].tolist())
                ys.extend(eye[:,1].tolist())
                break
        except Exception:
            pass
    return xs, ys


def load_salicon(img_dir, fix_dir, label=''):
    img_paths = sorted(glob.glob(os.path.join(img_dir,'*.jpg')) +
                       glob.glob(os.path.join(img_dir,'*.JPEG')))
    mat_paths = sorted(glob.glob(os.path.join(fix_dir,'*.mat')))
    img_dict  = {os.path.splitext(os.path.basename(p))[0]: p for p in img_paths}
    mat_dict  = {os.path.splitext(os.path.basename(p))[0]: p for p in mat_paths}
    common    = sorted(set(img_dict) & set(mat_dict))
    imgs, fix = [], []
    for stem in common:
        xs,ys = _read_salicon_mat(mat_dict[stem])
        if xs:
            imgs.append(img_dict[stem])
            fix.append({'x':xs,'y':ys})
    print(f'SALICON {label}: {len(imgs)} anh co fixation')
    return imgs, fix


def load_mit1003(data_dir, subjects, stim_dir):
    """Load MIT1003. Tim anh trong stim_dir, tong hop fixation tu 15 subjects."""
    first_dir = os.path.join(data_dir, subjects[0])
    mat_files = sorted(glob.glob(os.path.join(first_dir,'i*.mat')))
    img_ids   = [os.path.splitext(os.path.basename(f))[0] for f in mat_files]

    stim_dict = {}
    for ext in ['.jpeg','.jpg','.png','.JPEG','.JPG']:
        for p in glob.glob(os.path.join(stim_dir,'*'+ext)):
            stim_dict[os.path.splitext(os.path.basename(p))[0]] = p
    if not stim_dict:
        for ext in ['*.jpeg','*.jpg','*.png']:
            for p in glob.glob(os.path.join(stim_dir,'**',ext), recursive=True):
                stim_dict[os.path.splitext(os.path.basename(p))[0]] = p
    print(f'  Anh trong stim_dir: {len(stim_dict)}')

    imgs, fix_data, missing, no_fix = [], [], 0, 0
    for img_id in img_ids:
        img_path = stim_dict.get(img_id)
        if img_path is None:
            missing += 1
            continue
        all_x, all_y = [], []
        for subj in subjects:
            mp = os.path.join(data_dir, subj, img_id+'.mat')
            if not os.path.exists(mp): continue
            xs, ys = _read_mit_mat(mp)
            all_x.extend(xs)
            all_y.extend(ys)
        if all_x:
            imgs.append(img_path)
            fix_data.append({'x': all_x, 'y': all_y})
        else:
            no_fix += 1

    print(f'MIT1003: {len(imgs)} anh co fixation')
    if missing: print(f'  {missing} anh khong tim thay jpg')
    if no_fix:  print(f'  {no_fix} anh co jpg nhung fixation rong')
    return imgs, fix_data


def compute_baseline_ll(centerbias_template, fix_data, img_paths, downsample):
    """Baseline LL tu centerbias. Don vi: nats."""
    lls = []
    for fp, fix in zip(img_paths, fix_data):
        try: w, h = Image.open(fp).size
        except: continue
        h_ = int(h/downsample); w_ = int(w/downsample)
        cb = zoom(centerbias_template,
                  (h_/centerbias_template.shape[0], w_/centerbias_template.shape[1]),
                  order=0, mode='nearest')
        cb -= logsumexp(cb)
        xs = np.array(fix['x'], dtype=float)
        ys = np.array(fix['y'], dtype=float)
        # Loc NaN
        valid = ~(np.isnan(xs) | np.isnan(ys))
        xs, ys = xs[valid], ys[valid]
        xs = np.clip((xs/downsample).astype(int), 0, w_-1)
        ys = np.clip((ys/downsample).astype(int), 0, h_-1)
        if len(xs): lls.append(cb[ys,xs].mean())
    return float(np.mean(lls)) if lls else 0.0


print('Dataset OK (v5)')
print('fixation_mask: BINARY (0/1) — dung unique positions')


Dataset OK (v5)
fixation_mask: BINARY (0/1) — dung unique positions


---
## Cell 10 — Training Functions & Checkpoint Manager

In [10]:
# ============================================================
# CELL 10: TRAINING FUNCTIONS
# Checkpoint lưu cả local (/content/) lẫn Drive để không mất
# khi session Colab reset
# ============================================================

import shutil
from tqdm.notebook import tqdm


def train_one_epoch(model, loader, optimizer, device):
    model.train()
    losses, wts = [], []
    pbar = tqdm(loader, desc='Train', leave=False)
    for batch in pbar:
        optimizer.zero_grad()
        img  = batch['image'].to(device)
        cb   = batch['centerbias'].to(device)
        mask = batch['fixation_mask'].to(device)
        w    = batch['weight'].to(device)
        loss = -log_likelihood(model(img, cb), mask, weights=w)
        loss.backward()
        optimizer.step()
        losses.append(loss.item())
        wts.append(w.sum().item())
        pbar.set_postfix(loss=f'{np.average(losses,weights=wts):.4f}')
    return np.average(losses, weights=wts)


@torch.no_grad()
def evaluate(model, loader, baseline_ll, device):
    model.eval()
    lls, nsss, wts = [], [], []
    for batch in tqdm(loader, desc='Eval ', leave=False):
        img  = batch['image'].to(device)
        cb   = batch['centerbias'].to(device)
        mask = batch['fixation_mask'].to(device)
        w    = batch['weight'].to(device)
        out  = model(img, cb)
        lls.append( log_likelihood(out, mask, weights=w).item())
        nsss.append(nss(out, mask, weights=w).item())
        wts.append(w.sum().item())
    avg_ll  = np.average(lls,  weights=wts)
    avg_nss = np.average(nsss, weights=wts)
    ig_nats = avg_ll - baseline_ll
    ig_bits = ig_nats / np.log(2)
    return {'LL':avg_ll, 'IG_nats':ig_nats, 'IG_bits':ig_bits, 'NSS':avg_nss}


def save_checkpoint(model, optimizer, scheduler, epoch, metrics, name):
    # Đảm bảo folder tồn tại ngay trước khi lưu
    os.makedirs(CFG['local_ckpt'], exist_ok=True)
    os.makedirs(CFG['drive_ckpt'], exist_ok=True)
    state = {'model':model.state_dict(), 'optimizer':optimizer.state_dict(),
             'scheduler':scheduler.state_dict(), 'epoch':epoch, 'metrics':metrics}
    lp = os.path.join(CFG['local_ckpt'], name)
    dp = os.path.join(CFG['drive_ckpt'], name)
    torch.save(state, lp)
    shutil.copy2(lp, dp)
    print(f'  💾 local: {lp}')
    print(f'  ☁️  Drive: {dp}')


def load_checkpoint(model, name, optimizer=None, scheduler=None):
    lp = os.path.join(CFG['local_ckpt'], name)
    dp = os.path.join(CFG['drive_ckpt'], name)
    path = lp if os.path.exists(lp) else (dp if os.path.exists(dp) else None)
    if path is None:
        print(f'  Không tìm thấy: {name}')
        return 0, {}
    ckpt = torch.load(path, map_location=DEVICE, weights_only=False)
    model.load_state_dict(ckpt['model'])
    if optimizer  and 'optimizer'  in ckpt: optimizer.load_state_dict(ckpt['optimizer'])
    if scheduler  and 'scheduler'  in ckpt: scheduler.load_state_dict(ckpt['scheduler'])
    print(f'  ✅ Loaded {name} (epoch {ckpt["epoch"]})')
    return ckpt['epoch'], ckpt.get('metrics',{})


print('✅ Training functions OK')


✅ Training functions OK


---
## Cell 11 — Khởi tạo Model & Load Data

In [11]:
# ============================================================
# CELL 11: KHỞI TẠO MODEL & LOAD DATA
# Tự động dùng BACKBONE_INDICES và EXPERIMENT_NAME từ Cell 3
# ============================================================

import urllib.request
import numpy as np

# 11.1 Centerbias
CB_PATH = '/content/centerbias_mit1003.npy'
if not os.path.exists(CB_PATH):
    print('Downloading centerbias...')
    urllib.request.urlretrieve(
        'https://github.com/matthias-k/DeepGaze/releases/download/v1.0.0/centerbias_mit1003.npy',
        CB_PATH)
centerbias_template = np.load(CB_PATH)
print(f'Centerbias: {centerbias_template.shape}')

# 11.2 Build model theo BACKBONE_INDICES của experiment hiện tại
print(f'\nBuilding model: {CFG["experiment"]}')
print(f'Backbones: {[BACKBONES_CFG[i]["name"] for i in CFG["backbone_indices"]]}')
model = build_deepgaze_iie(
    backbone_indices=CFG['backbone_indices'],
    n_components=CFG['n_components']
).to(DEVICE)

# 11.3 SALICON
print('\nLoading SALICON...')
sal_imgs_tr, sal_fix_tr = load_salicon(SAL_IMG_TRAIN, SAL_FIX_TRAIN, 'train')
sal_imgs_va, sal_fix_va = load_salicon(SAL_IMG_VAL,   SAL_FIX_VAL,   'val')
if not sal_imgs_tr:
    raise RuntimeError('SALICON train rỗng! Kiểm tra Cell 2 và Cell 3')
ds_sal       = CFG['ds_salicon']
sal_baseline = compute_baseline_ll(centerbias_template, sal_fix_va, sal_imgs_va, ds_sal)
print(f'SALICON val baseline: {sal_baseline:.4f} nats ({sal_baseline/np.log(2):.4f} bits)')
sal_train_dl = DataLoader(
    SaliencyDataset(sal_imgs_tr, sal_fix_tr, centerbias_template, ds_sal),
    batch_size=CFG['batch_size'], shuffle=True,
    num_workers=CFG['num_workers'], collate_fn=collate_fn, pin_memory=True)
sal_val_dl = DataLoader(
    SaliencyDataset(sal_imgs_va, sal_fix_va, centerbias_template, ds_sal),
    batch_size=CFG['batch_size'], shuffle=False,
    num_workers=CFG['num_workers'], collate_fn=collate_fn, pin_memory=True)
print(f'SALICON: train={len(sal_imgs_tr)}, val={len(sal_imgs_va)}')

# 11.4 MIT1003
print('\nLoading MIT1003...')
mit_imgs, mit_fix = load_mit1003(MIT_DATA_DIR, MIT_SUBJECTS, MIT_STIM_DIR)
HAS_MIT = len(mit_imgs) > 0
if HAS_MIT:
    n_val        = max(1, len(mit_imgs)//10)
    mit_imgs_tr  = mit_imgs[:-n_val];  mit_fix_tr = mit_fix[:-n_val]
    mit_imgs_va  = mit_imgs[-n_val:];  mit_fix_va = mit_fix[-n_val:]
    ds_mit       = CFG['ds_mit']
    mit_baseline = compute_baseline_ll(centerbias_template, mit_fix_va, mit_imgs_va, ds_mit)
    print(f'MIT1003 val baseline: {mit_baseline:.4f} nats ({mit_baseline/np.log(2):.4f} bits)')
    mit_train_dl = DataLoader(
        SaliencyDataset(mit_imgs_tr, mit_fix_tr, centerbias_template, ds_mit),
        batch_size=CFG['batch_size'], shuffle=True,
        num_workers=CFG['num_workers'], collate_fn=collate_fn, pin_memory=True)
    mit_val_dl = DataLoader(
        SaliencyDataset(mit_imgs_va, mit_fix_va, centerbias_template, ds_mit),
        batch_size=CFG['batch_size'], shuffle=False,
        num_workers=CFG['num_workers'], collate_fn=collate_fn, pin_memory=True)
    print(f'MIT1003: train={len(mit_imgs_tr)}, val={len(mit_imgs_va)}')
else:
    print('MIT zip không có ảnh → chỉ chạy Phase 1')

print(f'\n✅ Cell 11 xong — Experiment: {CFG["experiment"]}')
print(f'   Checkpoint dir: {CFG["drive_ckpt"]}')


Centerbias: (1024, 1024)

Building model: SN_EN_DN
Backbones: ['ShapeNetC', 'EfficientNetB5', 'DenseNet201']
Building DeepGaze IIE:
  Backbones  : ['ShapeNetC', 'EfficientNetB5', 'DenseNet201']
  Components : 10 per backbone  |  Total: 30 sub-models
  Loading ShapeNetC... 

/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=None`.
  warnings.warn(msg)


Downloading: "https://bitbucket.org/robert_geirhos/texture-vs-shape-pretrained-models/raw/60b770e128fffcbd8562a3ab3546c1a735432d03/resnet50_finetune_60_epochs_lr_decay_after_30_start_resnet50_train_45_epochs_combined_IN_SF-ca06340c.pth.tar" to /root/.cache/torch/hub/checkpoints/resnet50_finetune_60_epochs_lr_decay_after_30_start_resnet50_train_45_epochs_combined_IN_SF-ca06340c.pth.tar


100%|██████████| 195M/195M [00:11<00:00, 17.1MB/s]


✅
  Loading EfficientNetB5... Downloading: "https://github.com/lukemelas/EfficientNet-PyTorch/releases/download/1.0/efficientnet-b5-b6417697.pth" to /root/.cache/torch/hub/checkpoints/efficientnet-b5-b6417697.pth


...

Loaded pretrained weights for efficientnet-b5
✅
  Loading DenseNet201... Downloading: "https://github.com/pytorch/vision/zipball/v0.10.0" to /root/.cache/torch/hub/v0.10.0.zip


/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=DenseNet201_Weights.IMAGENET1K_V1`. You can also use `weights=DenseNet201_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Downloading: "https://download.pytorch.org/models/densenet201-c1103571.pth" to /root/.cache/torch/hub/checkpoints/densenet201-c1103571.pth


100%|██████████| 77.4M/77.4M [00:00<00:00, 189MB/s]


✅

Parameters: 735,590 trainable / 76,696,334 total

Loading SALICON...
SALICON train: 10000 anh co fixation
SALICON val: 5000 anh co fixation
SALICON val baseline: -11.3537 nats (-16.3799 bits)
SALICON: train=10000, val=5000

Loading MIT1003...
  Anh trong stim_dir: 1003
MIT1003: 999 anh co fixation
MIT1003 val baseline: -11.6078 nats (-16.7465 bits)
MIT1003: train=900, val=99

✅ Cell 11 xong — Experiment: SN_EN_DN
   Checkpoint dir: /content/drive/MyDrive/Data PBL4/experiments/SN_EN_DN


In [12]:
# DEBUG CELL: Xem cấu trúc DATA lồng nhau
import scipy.io as sio
import numpy as np

mat_path = '/content/data/MIT/DATA/CNG/i05june05_static_street_boston_p1010764.mat'
mat = sio.loadmat(mat_path, squeeze_me=True, struct_as_record=False)

key = 'i05june05_static_street_boston_p1010764'
v   = mat[key]         # struct ngoài
DATA = v.DATA          # struct trong

print(f'DATA type      : {type(DATA).__name__}')
print(f'DATA._fieldnames: {DATA._fieldnames}')
print()

for fname in (DATA._fieldnames or []):
    try:
        val = getattr(DATA, fname)
        arr = np.array(val)
        print(f'  DATA.{fname}: type={type(val).__name__}, shape={arr.shape}, dtype={arr.dtype}')
        if arr.size <= 20:
            print(f'    value = {arr}')
        else:
            print(f'    sample = {arr.flat[:6]}')
    except Exception as e:
        print(f'  DATA.{fname}: ERROR {e}')

DATA type      : mat_struct
DATA._fieldnames: ['block', 'trial', 'eyeData']

  DATA.block: type=int, shape=(), dtype=int64
    value = 1
  DATA.trial: type=int, shape=(), dtype=int64
    value = 324
  DATA.eyeData: type=ndarray, shape=(725, 2), dtype=float64
    sample = [496.6835443 377.664     496.6835443 379.776     496.6835443 379.776    ]


---
## Cell 12 — Phase 1: Pretrain trên SALICON
> **7 epochs tổng** (milestones=[3,5]): LR 1e-3 → 1e-4 → 1e-5 → 1e-6 → stop  
> IG kỳ vọng: **0.25–0.40 nats/fix** (≈ 0.36–0.58 bits/fix)

In [13]:
# ============================================================
# CELL 12: PHASE 1 — PRETRAIN SALICON
# - Resume tự động từ checkpoint của experiment hiện tại
# - Checkpoint lưu vào folder riêng theo EXPERIMENT_NAME
# - milestones=[3,5,7] → 8 epochs tổng
# ============================================================

import glob as _glob

optimizer_sal = torch.optim.Adam(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=CFG['lr_pretrain'])
scheduler_sal = torch.optim.lr_scheduler.MultiStepLR(
    optimizer_sal, milestones=[3,5,7], gamma=CFG['lr_decay_factor'])

start_epoch = 0
best_ig     = -np.inf
history_sal = []
MAX_EPOCHS  = 8

# Resume từ checkpoint epoch cao nhất của experiment này
epoch_ckpts = sorted(
    _glob.glob(os.path.join(CFG['local_ckpt'], 'salicon_epoch*.pth')) +
    _glob.glob(os.path.join(CFG['drive_ckpt'], 'salicon_epoch*.pth')))

if epoch_ckpts:
    latest = epoch_ckpts[-1]
    print(f'[{CFG["experiment"]}] Resume: {os.path.basename(latest)}')
    start_epoch, prev_mets = load_checkpoint(
        model, os.path.basename(latest), optimizer_sal, scheduler_sal)
    best_ig = prev_mets.get('IG_nats', -np.inf)
    for _ in range(start_epoch):
        scheduler_sal.step()
    print(f'  Tiếp từ epoch {start_epoch+1} | LR={optimizer_sal.param_groups[0]["lr"]:.0e}')
    print(f'  Best IG: {best_ig:.4f} nats ({best_ig/np.log(2):.4f} bits)')
elif os.path.exists(os.path.join(CFG['drive_ckpt'],'best_salicon.pth')):
    load_checkpoint(model, 'best_salicon.pth')
    print(f'[{CFG["experiment"]}] Loaded best_salicon.pth')

if start_epoch >= MAX_EPOCHS:
    print(f'\n[{CFG["experiment"]}] Phase 1 đã hoàn thành {MAX_EPOCHS} epochs!')
    print(f'Best IG: {best_ig:.4f} nats ({best_ig/np.log(2):.4f} bits)')
elif optimizer_sal.param_groups[0]['lr'] < CFG['min_lr']:
    print(f'\n[{CFG["experiment"]}] LR < min_lr → đã hội tụ!')
else:
    print(f'\n=== PHASE 1: {CFG["experiment"]} ===')
    print(f'Backbones : {[BACKBONES_CFG[i]["name"] for i in CFG["backbone_indices"]]}')
    print(f'Epochs    : {start_epoch+1} → {MAX_EPOCHS} | LR={optimizer_sal.param_groups[0]["lr"]:.0e}\n')

    epoch = start_epoch
    while optimizer_sal.param_groups[0]['lr'] >= CFG['min_lr'] and epoch < MAX_EPOCHS:
        epoch += 1
        lr = optimizer_sal.param_groups[0]['lr']
        print(f'── Epoch {epoch}/{MAX_EPOCHS} | LR={lr:.0e} ──')

        loss = train_one_epoch(model, sal_train_dl, optimizer_sal, DEVICE)
        mets = evaluate(model, sal_val_dl, sal_baseline, DEVICE)

        ig_n, ig_b = mets['IG_nats'], mets['IG_bits']
        print(f'  Loss: {loss:.4f} | LL: {mets["LL"]:.4f}')
        print(f'  IG  : {ig_n:.4f} nats = {ig_b:.4f} bits')
        print(f'  NSS : {mets["NSS"]:.4f}')

        history_sal.append({'epoch':epoch,'loss':loss,**mets})
        save_checkpoint(model, optimizer_sal, scheduler_sal,
                        epoch, mets, f'salicon_epoch{epoch}.pth')
        if ig_n > best_ig:
            best_ig = ig_n
            save_checkpoint(model, optimizer_sal, scheduler_sal,
                            epoch, mets, 'best_salicon.pth')
            print(f'  ⭐ Best IG = {ig_n:.4f} nats ({ig_b:.4f} bits)')
        print()
        scheduler_sal.step()

    print(f'✅ Phase 1 session xong: epoch {epoch}/{MAX_EPOCHS}')
    print(f'   Best IG: {best_ig:.4f} nats ({best_ig/np.log(2):.4f} bits)')
    if epoch < MAX_EPOCHS:
        print(f'   Còn {MAX_EPOCHS-epoch} epochs → chạy lại Cell 12')


[SN_EN_DN] Resume: salicon_epoch8.pth
  ✅ Loaded salicon_epoch8.pth (epoch 8)
  Tiếp từ epoch 9 | LR=1e-06
  Best IG: 0.5528 nats (0.7976 bits)

[SN_EN_DN] Phase 1 đã hoàn thành 8 epochs!
Best IG: 0.5528 nats (0.7976 bits)


---
## Cell 13 — Phase 2: Finetune trên MIT1003
> Chỉ chạy nếu `HAS_MIT = True` (cần ALLSTIMULI.zip đã giải nén)  
> **6 epochs tổng** (milestones=[2,4])  
> IG kỳ vọng: **0.65–0.85 nats/fix** (≈ 0.94–1.23 bits/fix)

In [14]:
# ============================================================
# CELL 13: PHASE 2 — FINETUNE MIT1003
# - Resume tự động từ checkpoint của experiment hiện tại
# - Checkpoint lưu vào folder riêng theo EXPERIMENT_NAME
# - milestones=[2,4,6] → 8 epochs tổng
# ============================================================

import glob as _glob

if not HAS_MIT:
    print(f'[{CFG["experiment"]}] HAS_MIT=False → bỏ qua Phase 2')
else:
    optimizer_mit = torch.optim.Adam(
        filter(lambda p: p.requires_grad, model.parameters()),
        lr=CFG['lr_finetune'])
    scheduler_mit = torch.optim.lr_scheduler.MultiStepLR(
        optimizer_mit, milestones=[2,4,6], gamma=CFG['lr_decay_factor'])

    start_epoch = 0
    best_ig     = -np.inf
    history_mit = []
    MAX_EPOCHS  = 8

    # Resume từ checkpoint MIT của experiment này
    epoch_ckpts = sorted(
        _glob.glob(os.path.join(CFG['local_ckpt'], 'mit_epoch*.pth')) +
        _glob.glob(os.path.join(CFG['drive_ckpt'], 'mit_epoch*.pth')))

    if epoch_ckpts:
        latest = epoch_ckpts[-1]
        print(f'[{CFG["experiment"]}] Resume: {os.path.basename(latest)}')
        start_epoch, prev_mets = load_checkpoint(
            model, os.path.basename(latest), optimizer_mit, scheduler_mit)
        best_ig = prev_mets.get('IG_nats', -np.inf)
        for _ in range(start_epoch):
            scheduler_mit.step()
        print(f'  Tiếp từ epoch {start_epoch+1} | LR={optimizer_mit.param_groups[0]["lr"]:.0e}')
        print(f'  Best IG: {best_ig:.4f} nats ({best_ig/np.log(2):.4f} bits)')
    else:
        # Load best_salicon của experiment này làm starting point
        print(f'[{CFG["experiment"]}] Chưa có MIT checkpoint → load best_salicon.pth')
        load_checkpoint(model, 'best_salicon.pth')
        print(f'  Bắt đầu Phase 2 từ epoch 1/{MAX_EPOCHS}')

    if start_epoch >= MAX_EPOCHS:
        print(f'\n[{CFG["experiment"]}] Phase 2 đã hoàn thành {MAX_EPOCHS} epochs!')
        print(f'Best IG: {best_ig:.4f} nats ({best_ig/np.log(2):.4f} bits)')
    elif optimizer_mit.param_groups[0]['lr'] < CFG['min_lr']:
        print(f'\n[{CFG["experiment"]}] LR < min_lr → đã hội tụ!')
    else:
        print(f'\n=== PHASE 2: {CFG["experiment"]} ===')
        print(f'Backbones : {[BACKBONES_CFG[i]["name"] for i in CFG["backbone_indices"]]}')
        print(f'Dataset   : {len(mit_imgs_tr)} train | {len(mit_imgs_va)} val')
        print(f'Epochs    : {start_epoch+1} → {MAX_EPOCHS} | LR={optimizer_mit.param_groups[0]["lr"]:.0e}\n')

        epoch = start_epoch
        while optimizer_mit.param_groups[0]['lr'] >= CFG['min_lr'] and epoch < MAX_EPOCHS:
            epoch += 1
            lr = optimizer_mit.param_groups[0]['lr']
            print(f'── Epoch {epoch}/{MAX_EPOCHS} | LR={lr:.0e} ──')

            loss = train_one_epoch(model, mit_train_dl, optimizer_mit, DEVICE)
            mets = evaluate(model, mit_val_dl, mit_baseline, DEVICE)

            ig_n, ig_b = mets['IG_nats'], mets['IG_bits']
            nss_       = mets['NSS']
            print(f'  Loss: {loss:.4f} | LL: {mets["LL"]:.4f}')
            print(f'  IG  : {ig_n:.4f} nats = {ig_b:.4f} bits')
            print(f'  NSS : {nss_:.4f} {"OK" if 1.0 < nss_ < 5.0 else "CHECK"}')

            history_mit.append({'epoch':epoch,'loss':loss,**mets})
            save_checkpoint(model, optimizer_mit, scheduler_mit,
                            epoch, mets, f'mit_epoch{epoch}.pth')
            if ig_n > best_ig:
                best_ig = ig_n
                save_checkpoint(model, optimizer_mit, scheduler_mit,
                                epoch, mets, 'best_mit1003.pth')
                print(f'  ⭐ Best IG = {ig_n:.4f} nats ({ig_b:.4f} bits)')
            print()
            scheduler_mit.step()

        print(f'✅ Phase 2 session xong: epoch {epoch}/{MAX_EPOCHS}')
        print(f'   Best IG: {best_ig:.4f} nats ({best_ig/np.log(2):.4f} bits)')
        if epoch < MAX_EPOCHS:
            print(f'   Còn {MAX_EPOCHS-epoch} epochs → chạy lại Cell 13')
        else:
            print(f'   Hoàn thành! Kỳ vọng 2BB: 0.85-0.95 bits | 4BB: 1.13 bits')


[SN_EN_DN] Resume: mit_epoch2.pth
  ✅ Loaded mit_epoch2.pth (epoch 2)
  Tiếp từ epoch 3 | LR=1e-04
  Best IG: 0.5780 nats (0.8339 bits)

=== PHASE 2: SN_EN_DN ===
Backbones : ['ShapeNetC', 'EfficientNetB5', 'DenseNet201']
Dataset   : 900 train | 99 val
Epochs    : 3 → 8 | LR=1e-04

── Epoch 3/8 | LR=1e-04 ──


Train:   0%|          | 0/450 [00:00<?, ?it/s]

Eval :   0%|          | 0/50 [00:00<?, ?it/s]

  Loss: 10.8388 | LL: -11.0306
  IG  : 0.5772 nats = 0.8327 bits
  NSS : 2.1955 OK
  💾 local: /content/checkpoints_SN_EN_DN/mit_epoch3.pth
  ☁️  Drive: /content/drive/MyDrive/Data PBL4/experiments/SN_EN_DN/mit_epoch3.pth

── Epoch 4/8 | LR=1e-05 ──


Train:   0%|          | 0/450 [00:00<?, ?it/s]

...

Train:   0%|          | 0/450 [00:00<?, ?it/s]

Eval :   0%|          | 0/50 [00:00<?, ?it/s]

  Loss: 10.8228 | LL: -11.0328
  IG  : 0.5750 nats = 0.8295 bits
  NSS : 2.1929 OK
  💾 local: /content/checkpoints_SN_EN_DN/mit_epoch6.pth
  ☁️  Drive: /content/drive/MyDrive/Data PBL4/experiments/SN_EN_DN/mit_epoch6.pth

── Epoch 7/8 | LR=1e-06 ──


Train:   0%|          | 0/450 [00:00<?, ?it/s]

KeyboardInterrupt: 

---
## Cell 14 — Visualize Training & Predictions

In [ ]:
# ============================================================
# CELL 14: VISUALIZE
# 14.1: Training curves (Loss, IG, NSS)
# 14.2: Saliency map predictions trên ảnh val
# ============================================================

import matplotlib.pyplot as plt

# 14.1 Training curves
histories = []
try:    histories.append((history,     'SALICON pretrain', 'steelblue'))
except: pass
try:    histories.append((history_mit, 'MIT1003 finetune', 'tomato'))
except: pass

if histories:
    fig, axes = plt.subplots(1,3, figsize=(15,4))
    fig.suptitle('Training History', fontsize=13)
    for hist, label, color in histories:
        ep = [h['epoch'] for h in hist]
        for ax, key, title in zip(axes,
                ['loss','IG_nats','NSS'],
                ['Train Loss (nats)','Info Gain (nats/fix)','NSS']):
            ax.plot(ep, [h[key] for h in hist], 'o-', label=label, color=color)
            ax.set_title(title); ax.set_xlabel('Epoch')
            ax.legend(); ax.grid(alpha=0.3)
    plt.tight_layout()
    plot_path = os.path.join(CFG['drive_ckpt'], 'training_curves.png')
    plt.savefig(plot_path, dpi=120, bbox_inches='tight')
    plt.show()
    print(f'Saved → {plot_path}')

# 14.2 Predictions
best_ckpt = 'best_mit1003.pth' if HAS_MIT else 'best_salicon.pth'
load_checkpoint(model, best_ckpt)
model.eval()

val_imgs = mit_imgs_va if HAS_MIT else sal_imgs_va
val_fix  = mit_fix_va  if HAS_MIT else sal_fix_va
ds_vis   = CFG['ds_mit'] if HAS_MIT else CFG['ds_salicon']

for i in range(min(3, len(val_imgs))):
    img_orig = np.array(Image.open(val_imgs[i]).convert('RGB'))
    nh, nw   = int(img_orig.shape[0]/ds_vis), int(img_orig.shape[1]/ds_vis)
    img_rs   = np.array(Image.fromarray(img_orig).resize((nw,nh), Image.BILINEAR))
    cb       = zoom(centerbias_template,
                    (nh/centerbias_template.shape[0], nw/centerbias_template.shape[1]),
                    order=0, mode='nearest')
    cb -= logsumexp(cb)
    img_t = torch.from_numpy(img_rs.transpose(2,0,1).astype(np.float32)).unsqueeze(0).to(DEVICE)
    cb_t  = torch.from_numpy(cb.astype(np.float32)).unsqueeze(0).to(DEVICE)
    with torch.no_grad():
        ld = model(img_t, cb_t)[0,0].cpu().numpy()
    density = np.exp(ld)
    density = (density-density.min())/(density.max()-density.min()+1e-8)
    density = zoom(density, (img_orig.shape[0]/density.shape[0],
                             img_orig.shape[1]/density.shape[1]), order=1)
    fix = val_fix[i]
    fig, axes = plt.subplots(1,3, figsize=(14,4))
    axes[0].imshow(img_orig);   axes[0].set_title('Image');       axes[0].axis('off')
    axes[1].imshow(density, cmap='jet'); axes[1].set_title('Saliency'); axes[1].axis('off')
    axes[2].imshow(img_orig)
    axes[2].imshow(density, cmap='jet', alpha=0.5)
    axes[2].scatter(fix['x'][:100], fix['y'][:100], c='white', s=8, alpha=0.4)
    axes[2].set_title(f'Overlay + Fixations (val #{i})'); axes[2].axis('off')
    plt.tight_layout(); plt.show()

print('✅ Visualization hoàn tất')


---
## Cell 15 — Lưu Final Model về Drive

In [ ]:
# ============================================================
# CELL 15: BACKUP FINAL MODEL VỀ DRIVE
# Chạy sau khi training xong
# ============================================================

import shutil

# Copy tất cả checkpoints local → Drive
for fname in os.listdir(CFG['local_ckpt']):
    src = os.path.join(CFG['local_ckpt'], fname)
    dst = os.path.join(CFG['drive_ckpt'], fname)
    shutil.copy2(src, dst)
    print(f'✅ {fname} ({os.path.getsize(dst)/1e6:.1f} MB) → Drive')

# Lưu state_dict riêng (nhỏ hơn, dễ load)
best = 'best_mit1003.pth' if HAS_MIT else 'best_salicon.pth'
load_checkpoint(model, best)
final_path = os.path.join(CFG['drive_ckpt'], 'deepgaze2e_final_weights.pth')
torch.save(model.state_dict(), final_path)
print(f'\n⭐ Final weights → {final_path}')
print(f'   Size: {os.path.getsize(final_path)/1e6:.1f} MB')
print('✅ Tất cả đã backup về Drive')


---
## Cell 14 — So sánh kết quả các Experiment
> Chạy sau khi tất cả tổ hợp đã train xong để so sánh IG, NSS

In [ ]:
# ============================================================
# CELL 14: SO SANH KET QUA TAT CA EXPERIMENTS
# Chay sau khi train xong tat ca to hop
# ============================================================

import glob, os, torch
import numpy as np

DRIVE_BASE = '/content/drive/MyDrive/Data PBL4/experiments'

# Danh sach tat ca experiment can so sanh
ALL_EXPERIMENTS = {
    # 2 trong 4 backbone
    'SN_EN'    : [0,1],
    'SN_DN'    : [0,2],
    'SN_RX'    : [0,3],
    'EN_DN'    : [1,2],
    'EN_RX'    : [1,3],
    'DN_RX'    : [2,3],
    # 3 trong 4 backbone
    'SN_EN_DN' : [0,1,2],
    'SN_EN_RX' : [0,1,3],
    'SN_DN_RX' : [0,2,3],
    'EN_DN_RX' : [1,2,3],
}

BACKBONE_NAMES = ['ShapeNetC','EfficientNetB5','DenseNet201','ResNext50']

print(f'{'Experiment':12s} | {'Backbones':40s} | {'IG_sal(bits)':12s} | {'IG_mit(bits)':12s} | {'NSS_mit':8s} | {'Status':10s}')
print('-'*110)

results = []
for exp_name, bb_idx in ALL_EXPERIMENTS.items():
    ckpt_dir = os.path.join(DRIVE_BASE, exp_name)
    bb_str   = '+'.join([BACKBONE_NAMES[i][:4] for i in bb_idx])

    ig_sal = ig_mit = nss_mit = float('nan')
    status = 'not started'

    # Doc best_salicon
    sal_path = os.path.join(ckpt_dir, 'best_salicon.pth')
    if os.path.exists(sal_path):
        try:
            ck = torch.load(sal_path, map_location='cpu', weights_only=False)
            m  = ck.get('metrics', {})
            ig_sal = m.get('IG_bits', float('nan'))
            status = f'Ph1 ep{ck.get("epoch","?")}'
        except: pass

    # Doc best_mit1003
    mit_path = os.path.join(ckpt_dir, 'best_mit1003.pth')
    if os.path.exists(mit_path):
        try:
            ck = torch.load(mit_path, map_location='cpu', weights_only=False)
            m  = ck.get('metrics', {})
            ig_mit  = m.get('IG_bits',  float('nan'))
            nss_mit = m.get('NSS',      float('nan'))
            status  = f'Ph2 ep{ck.get("epoch","?")}'
        except: pass

    ig_sal_str  = f'{ig_sal:.4f}'  if not np.isnan(ig_sal)  else '---'
    ig_mit_str  = f'{ig_mit:.4f}'  if not np.isnan(ig_mit)  else '---'
    nss_mit_str = f'{nss_mit:.4f}' if not np.isnan(nss_mit) else '---'

    print(f'{exp_name:12s} | {bb_str:40s} | {ig_sal_str:12s} | {ig_mit_str:12s} | {nss_mit_str:8s} | {status}')
    results.append({'exp':exp_name,'bb':bb_idx,'ig_sal':ig_sal,'ig_mit':ig_mit,'nss':nss_mit})

# Tim ket qua tot nhat
valid = [r for r in results if not np.isnan(r['ig_mit'])]
if valid:
    best = max(valid, key=lambda r: r['ig_mit'])
    print(f'\n⭐ Best experiment: {best["exp"]} | IG_MIT = {best["ig_mit"]:.4f} bits/fix')
    print(f'   Backbones: {[BACKBONE_NAMES[i] for i in best["bb"]]}')
    print(f'\nTham chieu paper (full 4BB): 1.13 bits/fix')
